### 00. import dependencies

In [26]:
import os
import warnings
from dotenv import load_dotenv
from typing import Literal,TypedDict, Annotated

# for routing
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# langchain
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, AnyMessage
from langchain_core.tools import tool

# langgraph
from langgraph.graph import StateGraph, START, END,add_messages
from langgraph.prebuilt import create_react_agent
from langgraph.types import Command
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg_pool import ConnectionPool

# DB + ORM
from pydantic import BaseModel,Field
from sqlalchemy import create_engine,Column,Integer,String
from sqlalchemy.orm import sessionmaker,declarative_base

# vector DB
import chromadb

warnings.filterwarnings("ignore")


### 01. Creating the Environment variables

In [11]:
#### Load the api_key

load_dotenv(dotenv_path='../.env')

API_KEY = os.getenv("GOOGLE_API_KEY")
DB_URL = os.getenv("DB_URI")
DEFAULT_GEMINI_MODEL = "gemini-2.5-flash"

if not API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")

if not DB_URL:
    raise ValueError("DB_URL not found in environment variables")

In [12]:
#### initialize the models

llm = ChatGoogleGenerativeAI(model=DEFAULT_GEMINI_MODEL, temperature=0.2)
embeddings = GoogleGenerativeAIEmbeddings(model="model/embedding-001")

### 02. RAG pipline

In [13]:
### initialize ChromaDB

chroma_client = chromadb.Client()
collection_name = "enterprise_hr_collection"

try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass
collection = chroma_client.create_collection(collection_name)

In [14]:
### Load the documents

file_paths = [
    "../data/raw/hr_policy.pdf",
    "../data/raw/complaints_policy.pdf",
    "../data/raw/leave_policy.pdf"
]

documents = []
for path in file_paths:
    if os.path.exists(path):
        loader = PyPDFLoader(path)
        documents.extend(loader.load())
    else:
        print(f"Warning: {path} not found. Skipping.")

In [15]:
### Chunking and embedding

if documents:
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ".", " "],
        chunk_size=400,
        chunk_overlap=50,
        length_function=len,
        is_separator_regex=False,
    )
    chunks =text_splitter.split_documents(documents)
    texts = [c.page_content for c in chunks]
    metadatas = [c.metadata for c in chunks]
    ids = [f"doc_{i}" for i in range(len(chunks))]

    collection.add(documents=texts, metadatas=metadatas, ids=ids)
    print(f"Added {len(texts)} chunks into chromaDB.")
else:
    print("No documents to add.")

Added 17 chunks into chromaDB.


### 03. database setup (sample)

In [19]:
Base =  declarative_base()

class EmployeeModel(Base):
    __tablename__ = "employees"
    employee_id = Column(String, primary_key=True)
    name = Column(String)
    pto_days = Column(Integer)
    sick_days = Column(Integer)
    email = Column(String)

engine = create_engine(DB_URL)
SessionLocal= sessionmaker(bind=engine)

Base.metadata.create_all(bind=engine)

### 04. Creating tools

In [23]:
@tool
def check_sql_leave_balance(employee_id:str)-> str:
    """Check PTO and sick leave balance for a given employee ID."""
    db = SessionLocal()
    try:
        emp = db.query(EmployeeModel).filter_by(employee_id = employee_id).first()
        if not emp:
            return f"Employee {employee_id} not found."
        return f"Employee {emp.name} has {emp.pto_days} PTO and {emp.sick_days} sick leaves."
    finally:
        db.close()


@tool
def hr_vector_search(query:str)->str:
    """Search HR policy documents using semantic vector search."""
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    if results['documents'] and results['documents'][0]:
        docs = results["documents"][0]
        metas = results["metadatas"][0]

        formatted_context = ""

        for i in range(len(docs)):
            source_path = metas[i].get('source', 'Unknown Document')
            filename = os.path.basename(source_path)
            page_num = metas[i].get('page', -1) + 1

            formatted_context += f"--- SOURCE: {filename} (Page {page_num}) ---\n{docs[i]}\n\n"
        return formatted_context

    return "No relevant policy found."

@tool
def draft_and_send_email(recipient: str, subject: str, body: str, is_confirmed: bool = False) -> str:
    """Draft an email and send it only after user confirmation."""
    if not is_confirmed:
        return f"""
EMAIL_DRAFT:
To: {recipient}
Subject: {subject}

{body}

Reply YES to confirm sending.
"""
    print(f"[EMAIL SENT] → {recipient}")
    return "EMAIL_SENT_SUCCESS"

### 05. Create router layer 1

In [27]:
### making state
class State[Typedict]:
    message: Annotated[list[AnyMessage],add_messages]
    user_context:dict


In [ ]:
intent_profile = {
    "leave_agent" : ["leave","vacation","pto","holiday"],
    "complaint_agent" : ["complaint","issue","harassment"],
    "hr_agent":["policy","salary","benefits","dress code"]
}

def semantic_router(state:State):
    text = state["message"][-1].content.lower()
    vec = TfidfVectorizer()
    intents = sum(intent_profile.values(), [])
    corpus = [text]+intents

    tfidf = vec.fit_transform(corpus)
    sim = cosine_similarity(tfidf[0:1], tfidf[1:])

    score = sim.max()
    idx = sim.argmax()

    if score < 0.2:
        return Command(goto="hr_agent")

    match = next(k for k,v in intent_profile.items() if intents[idx] in v)
    return Command(goto=match)